In [1]:
import os
import sys
from pathlib import Path

# Run from the repo root so `fpl_hq` imports and relative paths resolve wherever this notebook lives.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fpl_hq.paths import GAMEWEEKS_DIR, REFERENCE_DIR, gw_dir

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/augustustranter/Documents/Programming/Projects/fpl/fpl_agent


In [2]:
# This variable indicates the gameweek for which we are generating a report.
# E.g. if gameweek = 3, transcripts in data/2026-27/gameweeks/gw3/transcripts/ are analysed.
gameweek = 7

# 1. FILTER TEXT

In [3]:
import importlib.util
import sys
from pathlib import Path

filter_text_path = Path("fpl_hq/consensus/text/1_filter_text.py")
filter_spec = importlib.util.spec_from_file_location("text_filter", filter_text_path)
assert filter_spec is not None and filter_spec.loader is not None
filter_module = importlib.util.module_from_spec(filter_spec)
sys.modules[filter_spec.name] = filter_module
filter_spec.loader.exec_module(filter_module)
process_text_folder = filter_module.process_text_folder

text_cache_dir = gw_dir(gameweek) / "text_cache"
filter_cache_dir = text_cache_dir / "filter_cache"
segment_words = 10
max_workers = 4

filtered_text_sources = process_text_folder(
    gameweek=gameweek,
    cache_dir=filter_cache_dir,
    segment_words=segment_words,
    max_workers=max_workers,
    refresh=False,
)

filtered_text_payload = {
    source_name: filtered.text
    for source_name, filtered in filtered_text_sources.items()
}

print(f"Filter cache written to: {filter_cache_dir}")
print(f"Loaded {len(filtered_text_sources)} filtered text source(s) into memory")
print(f"Segment target word count: {segment_words}")
print(f"Parallel workers: {max_workers}")
print(sorted(filtered_text_payload))

Filter cache written to: gw/gw7/text_cache/filter_cache
Loaded 9 filtered text source(s) into memory
Segment target word count: 10
Parallel workers: 4
['fpl_gameweek_7_captain_picks_gross_haaland_html', 'fpl_gameweek_7_chip_strategy_wildcard_triple_captain_html', 'fpl_gameweek_7_tips_best_players_predicted_line_ups_team_news_more_4', 'fpl_gameweek_7_wildcard_draft_best_xi_html', 'fpl_player_price_changes_risers_fallers_and_top_transfers_for_gameweek_7', 'hincapie_saliba_latest_fpl_team_news_ahead_of_gameweek_7', 'tht_fpl_top_transfer_tips_gameweek_7', 'triple_captain_haaland_in_gameweek_7_how_and_when_to_use_your_fpl_chips', 'where_could_rotation_in_fpl_gameweeks_6_10']


# 2. MORE FILTERING

In [4]:
import importlib.util
import sys
from pathlib import Path

selection_text_path = Path("fpl_hq/consensus/text/2_selection_filter.py")
selection_spec = importlib.util.spec_from_file_location("text_selection", selection_text_path)
assert selection_spec is not None and selection_spec.loader is not None
selection_module = importlib.util.module_from_spec(selection_spec)
sys.modules[selection_spec.name] = selection_module
selection_spec.loader.exec_module(selection_module)
select_text_sources = selection_module.select_text_sources

text_selection_cache_dir = text_cache_dir / "selection_cache"
from fpl_hq.consensus.llm import MODEL as selection_model
selection_max_workers = 4

selected_text_sources = select_text_sources(
    filtered_text_sources,
    cache_dir=text_selection_cache_dir,
    model=selection_model,
    max_workers=selection_max_workers,
    refresh=False,
)

selected_text_payload = {
    source_name: selected.text
    for source_name, selected in selected_text_sources.items()
}

print(f"Selection cache written to: {text_selection_cache_dir}")
print(f"Loaded {len(selected_text_sources)} selected text source(s) into memory")
print(f"Selection model: {selection_model}")
print(f"Parallel workers: {selection_max_workers}")
print(sorted(selected_text_payload))

Selection cache written to: gw/gw7/text_cache/selection_cache
Loaded 9 selected text source(s) into memory
Selection model: gpt-4.1-mini
Parallel workers: 4
['fpl_gameweek_7_captain_picks_gross_haaland_html', 'fpl_gameweek_7_chip_strategy_wildcard_triple_captain_html', 'fpl_gameweek_7_tips_best_players_predicted_line_ups_team_news_more_4', 'fpl_gameweek_7_wildcard_draft_best_xi_html', 'fpl_player_price_changes_risers_fallers_and_top_transfers_for_gameweek_7', 'hincapie_saliba_latest_fpl_team_news_ahead_of_gameweek_7', 'tht_fpl_top_transfer_tips_gameweek_7', 'triple_captain_haaland_in_gameweek_7_how_and_when_to_use_your_fpl_chips', 'where_could_rotation_in_fpl_gameweeks_6_10']


# 3. EXTRACT TEXT

In [5]:
import importlib.util
import json
from pathlib import Path

extract_text_path = Path("fpl_hq/consensus/text/3_extract_text.py")
extract_spec = importlib.util.spec_from_file_location("text_extract", extract_text_path)
assert extract_spec is not None and extract_spec.loader is not None
extract_module = importlib.util.module_from_spec(extract_spec)
extract_spec.loader.exec_module(extract_module)
build_combined_output = extract_module.build_combined_output
process_filtered_text_sources = extract_module.process_filtered_text_sources

text_extraction_cache_dir = text_cache_dir / "extraction_cache"
text_extraction_output_path = Path(f"{GAMEWEEKS_DIR}/gw{gameweek}/data/gw_{gameweek}_text_extractions.json")
from fpl_hq.consensus.llm import MODEL as extraction_model
extraction_max_workers = 4
extraction_batch_size = 4

text_extraction_results = process_filtered_text_sources(
    selected_text_payload,
    cache_dir=text_extraction_cache_dir,
    model=extraction_model,
    max_workers=extraction_max_workers,
    batch_size=extraction_batch_size,
    refresh=False,
)

combined_text_extraction = build_combined_output(gameweek, text_extraction_results)
text_extraction_output_path.parent.mkdir(parents=True, exist_ok=True)
text_extraction_output_path.write_text(
    json.dumps(combined_text_extraction, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Extraction cache written to: {text_extraction_cache_dir}")
print(f"Combined extraction JSON written to: {text_extraction_output_path}")
print(f"Extracted {len(combined_text_extraction['recommendations'])} recommendation(s)")
print(f"Processed {len(text_extraction_results)} source(s) in batches of {extraction_batch_size}")

Extraction cache written to: gw/gw7/text_cache/extraction_cache
Combined extraction JSON written to: gw/gw7/data/gw_7_text_extractions.json
Extracted 40 recommendation(s)
Processed 9 source(s) in batches of 4
